# Exp1 (04): Richer behavioural features and ablation

Adds the behavioural indicators from the proposal that notebook 01 did not yet include, then repeats the
source ablation (as in notebook 03) with the richer features.

**New features**
* **POS/cash:** max days past due, months past due, average remaining instalments
* **Credit card:** months paid below the minimum, months past due, average minimum-payment ratio, balance trend
* **Instalments:** max and average days late, underpayment frequency, total paid / total due
* **Bureau:** max days overdue, active loan count
* **Previous applications:** approval rate, applications in the last year

Old features from notebook 01 are kept, so each group is a superset of the earlier one.
Settings are the same as notebook 03 (all-sources tuning from notebook 01).

## 1. Setup and existing features (same as notebook 01)

In [1]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

SHARED = Path("../hongwei_outputs/outputs").resolve()   # shared outputs from data cleaning
CLEAN = SHARED / "clean_data"
HERE = Path(".").resolve()                              # this notebook's folder (where prediction files go)

splits = pd.read_csv(CLEAN / "splits.csv")
print(splits["split"].value_counts().to_string())

split
train    69999
val      15001
test     15000


In [2]:
EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]

app = pd.read_parquet(CLEAN / "application.parquet").drop(columns=["TARGET"])
app = app.merge(splits[["SK_ID_CURR", "TARGET", "split"]], on="SK_ID_CURR", how="inner")
app_cols = [c for c in app.columns if c not in EXCLUDE]
print(f"{len(app):,} applicants, {len(app_cols)} application features")

100,000 applicants, 120 application features


In [3]:
# Bureau: loans at other lenders
b = pd.read_parquet(CLEAN / "bureau.parquet")
bureau = b.groupby("SK_ID_CURR").agg(
    n_bureau=("SK_ID_BUREAU", "size"),
    bureau_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_credit=("AMT_CREDIT_SUM", "sum"),
)
bureau["bureau_debt_ratio"] = bureau["bureau_debt"] / bureau["bureau_credit"].where(bureau["bureau_credit"] > 0)

# Previous Home Credit applications (duplicate submissions removed)
p = pd.read_parquet(CLEAN / "previous_application.parquet")
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["refused"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Refused").astype(int)
prev = p.groupby("SK_ID_CURR").agg(
    n_prev=("SK_ID_PREV", "size"),
    prev_refusal_rate=("refused", "mean"),
    prev_amt_mean=("AMT_APPLICATION", "mean"),
)

# Instalments: one row per instalment (split payments already merged in cleaning)
i = pd.read_parquet(CLEAN / "installments.parquet")
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]) > 0, np.nan)   # unpaid: not counted
i["under"] = (i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)).clip(lower=0)
inst = i.groupby("SK_ID_CURR").agg(
    n_inst=("SK_ID_PREV", "size"),
    late_rate=("late", "mean"),
    underpay_mean=("under", "mean"),
)

# Credit cards: utilisation
c = pd.read_parquet(CLEAN / "credit_card.parquet")
c["util"] = c["AMT_BALANCE"] / c["AMT_CREDIT_LIMIT_ACTUAL"].where(c["AMT_CREDIT_LIMIT_ACTUAL"] > 0)
cc = c.groupby("SK_ID_CURR").agg(cc_util_mean=("util", "mean"), cc_months=("MONTHS_BALANCE", "nunique"))

hist = bureau.join([prev, inst, cc], how="outer").reset_index()
hist_cols = [col for col in hist.columns if col != "SK_ID_CURR"]
print(f"History features: {len(hist_cols)} columns for {len(hist):,} applicants with any history")

History features: 13 columns for 99,277 applicants with any history


In [4]:
feat_old = app.merge(hist, on="SK_ID_CURR", how="left")
BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

## 2. New features

In [5]:
# POS / cash loans
q = pd.read_parquet(CLEAN / "pos_cash.parquet", columns=["SK_ID_CURR", "SK_DPD", "CNT_INSTALMENT_FUTURE"])
q["dpd_flag"] = (q["SK_DPD"] > 0).astype("int32")
pos = q.groupby("SK_ID_CURR").agg(
    pos_max_dpd=("SK_DPD", "max"),
    pos_months_dpd=("dpd_flag", "sum"),
    pos_remaining_inst=("CNT_INSTALMENT_FUTURE", "mean"),
)

# Credit card, month by month (one row per applicant-month, summed over contracts)
c = pd.read_parquet(CLEAN / "credit_card.parquet", columns=[
    "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_TOTAL_CURRENT", "SK_DPD"])
c["below_min"] = ((c["AMT_INST_MIN_REGULARITY"] > 0) &
                  (c["AMT_PAYMENT_TOTAL_CURRENT"].fillna(0) < c["AMT_INST_MIN_REGULARITY"])).astype("int32")
c["dpd_flag"] = (c["SK_DPD"] > 0).astype("int32")
c["min_ratio"] = c["AMT_PAYMENT_TOTAL_CURRENT"] / c["AMT_INST_MIN_REGULARITY"].where(c["AMT_INST_MIN_REGULARITY"] > 0)
m = c.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False).agg(
    bal=("AMT_BALANCE", "sum"), lim=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
    below=("below_min", "max"), dpd=("dpd_flag", "max"), min_ratio=("min_ratio", "mean"))
m["util"] = m["bal"] / m["lim"].where(m["lim"] > 0)
recent = m[m["MONTHS_BALANCE"] >= -3].groupby("SK_ID_CURR")["util"].mean()
card = m.groupby("SK_ID_CURR").agg(
    cc_months_below_min=("below", "sum"),
    cc_months_dpd=("dpd", "sum"),
    cc_min_ratio_mean=("min_ratio", "mean"),
    cc_util_overall=("util", "mean"),
)
card["cc_balance_trend"] = recent.reindex(card.index) - card["cc_util_overall"]
card = card.drop(columns="cc_util_overall")

# Instalments
i = pd.read_parquet(CLEAN / "installments.parquet", columns=[
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"])
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["days_late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]).clip(lower=0), np.nan)
i["under_flag"] = ((i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)) > 1.0).astype("int32")
i["paid_amt"] = i["AMT_PAYMENT"].fillna(0)
inst_new = i.groupby("SK_ID_CURR").agg(
    inst_max_days_late=("days_late", "max"),
    inst_mean_days_late=("days_late", "mean"),
    inst_underpay_freq=("under_flag", "mean"),
    paid_sum=("paid_amt", "sum"),
    due_sum=("AMT_INSTALMENT", "sum"),
)
inst_new["inst_paid_to_due"] = inst_new["paid_sum"] / inst_new["due_sum"].where(inst_new["due_sum"] > 0)
inst_new = inst_new.drop(columns=["paid_sum", "due_sum"])

# Bureau
b = pd.read_parquet(CLEAN / "bureau.parquet", columns=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "CREDIT_ACTIVE"])
bureau_new = b.groupby("SK_ID_CURR").agg(
    bureau_max_days_overdue=("CREDIT_DAY_OVERDUE", "max"),
    bureau_active_count=("CREDIT_ACTIVE", lambda s: (s.astype(str) == "Active").sum()),
)

# Previous applications
p = pd.read_parquet(CLEAN / "previous_application.parquet", columns=[
    "SK_ID_CURR", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "IS_DUPLICATE_SUBMISSION"])
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["approved"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Approved").astype("int32")
p["last_year"] = (p["DAYS_DECISION"] >= -365).astype("int32")
prev_new = p.groupby("SK_ID_CURR").agg(prev_approval_rate=("approved", "mean"), prev_apps_last_year=("last_year", "sum"))

new_hist = pos.join([card, inst_new, bureau_new, prev_new], how="outer").reset_index()
NEW_POS = ["pos_max_dpd", "pos_months_dpd", "pos_remaining_inst"]
NEW_CARD = ["cc_months_below_min", "cc_months_dpd", "cc_min_ratio_mean", "cc_balance_trend"]
NEW_INST = ["inst_max_days_late", "inst_mean_days_late", "inst_underpay_freq", "inst_paid_to_due"]
NEW_BUREAU = ["bureau_max_days_overdue", "bureau_active_count"]
NEW_PREV = ["prev_approval_rate", "prev_apps_last_year"]
assert set(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_PREV) == set(new_hist.columns) - {"SK_ID_CURR"}
print(f"New features: {len(new_hist.columns) - 1} columns for {len(new_hist):,} applicants")
display(new_hist.describe().T[["count", "mean", "min", "max"]].round(3))

New features: 15 columns for 99,290 applicants


,count,mean,min,max
SK_ID_CURR,99290.0,277928.762,100011.00,456248.000
pos_max_dpd,94164.0,15.262,0.00,4231.000
pos_months_dpd,94164.0,0.885,0.00,120.000
pos_remaining_inst,94155.0,9.215,0.00,59.500
cc_months_below_min,28368.0,2.096,0.00,70.000
cc_months_dpd,28368.0,1.546,0.00,94.000
cc_min_ratio_mean,19351.0,34.067,0.00,471288.531
cc_balance_trend,21320.0,0.007,-0.97,2.767
inst_max_days_late,94873.0,18.639,0.00,2852.000
inst_mean_days_late,94873.0,1.282,0.00,2813.000


## 3. Join all features and define the groups

In [6]:
feat = feat_old.merge(new_hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU + NEW_BUREAU,
    "+ Previous": app_cols + PREVIOUS + NEW_PREV,
    "+ Repayment": app_cols + REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
    "All sources": app_cols + hist_cols + NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_PREV,
}

## 4. Fit each group on train (same settings as notebook 03)

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"exp1_v2_{name.replace(' ', '_').replace('+', 'plus').lower()}.csv"
    out.to_csv(path, index=False)
    FILES[name] = path
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4}  -> {path.name}")

Application  inputs= 250  trees= 278  -> exp1_v2_application.csv


+ Bureau     inputs= 257  trees= 335  -> exp1_v2_plus_bureau.csv


+ Previous   inputs= 255  trees= 357  -> exp1_v2_plus_previous.csv


+ Repayment  inputs= 266  trees= 324  -> exp1_v2_plus_repayment.csv


All sources  inputs= 278  trees= 415  -> exp1_v2_all_sources.csv


## 5. Evaluate with the shared code (`eval_exp1.py`), 1000 bootstrap resamples

In [8]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = ev.compare_prediction_files([FILES[l] for l in labels], labels, n_boot=1000)
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application,0.156,0.741 (0.726-0.754),0.215 (0.197-0.236),0.228 (0.211-0.246),0.377 (0.350-0.403),0.284 (0.265-0.303),0.246 (0.224-0.268),0.305 (0.281-0.328)
+ Bureau,0.158,0.743 (0.729-0.757),0.224 (0.205-0.245),0.231 (0.214-0.249),0.377 (0.349-0.403),0.286 (0.266-0.305),0.255 (0.234-0.278),0.316 (0.294-0.339)
+ Previous,0.154,0.742 (0.727-0.756),0.222 (0.203-0.244),0.225 (0.208-0.243),0.384 (0.358-0.411),0.284 (0.265-0.303),0.247 (0.226-0.269),0.306 (0.282-0.329)
+ Repayment,0.154,0.755 (0.741-0.770),0.240 (0.219-0.264),0.237 (0.220-0.255),0.400 (0.373-0.425),0.298 (0.278-0.316),0.263 (0.242-0.285),0.325 (0.301-0.348)
All sources,0.142,0.758 (0.743-0.772),0.251 (0.230-0.274),0.228 (0.211-0.244),0.443 (0.414-0.471),0.301 (0.282-0.319),0.271 (0.249-0.294),0.336 (0.313-0.359)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,+ Bureau,Application,roc_auc,+0.0029 (-0.0001 to +0.0060),0.076,no significant difference
1,+ Bureau,Application,pr_auc,+0.0085 (+0.0013 to +0.0146),0.014,significantly better
2,+ Previous,Application,roc_auc,+0.0016 (-0.0018 to +0.0046),0.32,no significant difference
3,+ Previous,Application,pr_auc,+0.0070 (+0.0008 to +0.0122),0.022,significantly better
4,+ Repayment,Application,roc_auc,+0.0150 (+0.0101 to +0.0202),4.9e-09,significantly better
5,+ Repayment,Application,pr_auc,+0.0250 (+0.0148 to +0.0344),0,significantly better
6,All sources,Application,roc_auc,+0.0175 (+0.0120 to +0.0229),4.7e-09,significantly better
7,All sources,Application,pr_auc,+0.0359 (+0.0236 to +0.0462),0,significantly better


## 6. Save the applicant-level feature table (for teammates)

In [9]:
feat.drop(columns=["split", "TARGET"]).to_parquet(HERE / "exp1_features_v2.parquet", index=False)
print("saved exp1_features_v2.parquet", feat.shape)

saved exp1_features_v2.parquet (100000, 153)
